# 01 — Data Loading & Preprocessing
**Mohammad Taiyab Khan** | MSc Data Science & Analytics | Royal Holloway, University of London

This notebook covers:
1. Loading the TweetEval training dataset (recommended) or Sentiment140
2. Exploring and cleaning the data
3. Two text cleaning pipelines:
   - `clean_for_lr()` — stemming + stopwords removed (for TF-IDF + LR)
   - `clean_for_bert()` — light cleaning (for transformer/LLM models)
4. Train/test splitting
5. Saving cleaned data for use in notebook 02

---
**Output files produced:**
- `data/tweeteval_clean.csv` — cleaned TweetEval data, ready for model training
- `data/euro2024_clean.csv` — cleaned Euro 2024 domain tweets

In [1]:
# ── CELL 1: Install dependencies ─────────────────────────────────
# Only needed first run. Skip if already installed.
!pip install pandas numpy scikit-learn nltk vaderSentiment --quiet


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, pathlib
# Ensure all relative paths resolve from the project root, not the notebooks/ folder
project_root = pathlib.Path(__file__).parent.parent if '__file__' in dir() else pathlib.Path.cwd().parent
if (project_root / 'data').exists():
    os.chdir(project_root)
print(f'Working directory: {os.getcwd()}')

Working directory: D:\Projects\RHUL\PROJECT\Football_Sentiment_ResearchPaper


In [3]:
# ── CELL 2: Imports ───────────────────────────────────────────────
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print('Libraries loaded')

Libraries loaded


## Step 1 — Load Training Data

We use **TweetEval** (Barbieri et al., 2020) as our training dataset.
It has 57,763 labelled tweets across three classes: negative (0), neutral (1), positive (2).

This replaces Sentiment140 used in the original dissertation because:
- TweetEval has a proper neutral class (Sentiment140 is binary only)
- TweetEval is the current standard benchmark for Twitter NLP
- It allows direct comparison with published transformer baselines

> **To download TweetEval**, run the cell below OR download manually from:
> https://github.com/cardiffnlp/tweeteval — files are in `datasets/sentiment/`

In [4]:
# ── CELL 3: Load TweetEval ────────────────────────────────────────
# Option A: load from CSV (if you have tweeteval_training.csv already)
try:
    df = pd.read_csv('data/tweeteval_training.csv')
    print(f'Loaded from CSV: {len(df):,} tweets')

# Option B: download via HuggingFace datasets library
except FileNotFoundError:
    print('tweeteval_training.csv not found. Trying HuggingFace datasets...')
    try:
        from datasets import load_dataset
        raw = load_dataset('tweet_eval', 'sentiment')
        # Combine train + validation + test splits
        records = []
        for split in ['train', 'validation', 'test']:
            for item in raw[split]:
                records.append({'text': item['text'], 'label_3class': item['label']})
        df = pd.DataFrame(records)
        # Binary label (drop neutral for binary training)
        df['sentiment'] = df['label_3class'].map({0: 0, 1: None, 2: 1})
        df['source'] = 'tweeteval'
        df.to_csv('data/tweeteval_training.csv', index=False)
        print(f'Downloaded and saved: {len(df):,} tweets')
    except Exception as e:
        print(f'Could not download: {e}')
        print('Please download TweetEval manually — see notebook header for instructions.')

print(f'\nDataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

Loaded from CSV: 57,763 tweets

Dataset shape: (57763, 4)
Columns: ['text', 'label_3class', 'sentiment', 'source']


,text,label_3class,sentiment,source
0,"""QT @user In the original draft of the 7th boo...",2,1.0,tweeteval
1,"""Ben Smith / Smith (concussion) remains out of...",1,NaN,tweeteval
2,Sorry bout the stream last night I crashed out...,1,NaN,tweeteval


In [5]:
# ── CELL 4: Explore label distribution ───────────────────────────
print('Label distribution (3-class):')
label_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
dist = df['label_3class'].value_counts().sort_index()
for lbl, count in dist.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'  {label_names.get(lbl, lbl):<10} {count:>6,}  ({pct:>5.1f}%)  {bar}')

print(f'\nTotal: {len(df):,} tweets')
print(f'Binary subset (pos/neg only): {df["sentiment"].notna().sum():,} tweets')

Label distribution (3-class):
  Negative   11,051  ( 19.1%)  █████████
  Neutral    26,534  ( 45.9%)  ██████████████████████
  Positive   20,178  ( 34.9%)  █████████████████

Total: 57,763 tweets
Binary subset (pos/neg only): 31,229 tweets


In [6]:
# ── CELL 5: Check for missing values ─────────────────────────────
print('Missing values:')
print(df.isnull().sum())
print(f'\nEmpty text rows: {(df["text"].str.strip() == "").sum()}')

# Drop missing text
df = df.dropna(subset=['text'])
df = df[df['text'].str.strip() != '']
print(f'\nAfter dropping empty: {len(df):,} tweets')

Missing values:
text                0
label_3class        0
sentiment       26534
source              0
dtype: int64

Empty text rows: 0

After dropping empty: 57,763 tweets


## Step 2 — Text Cleaning

We define **two separate cleaning functions** because different models need different text formats:

| Function | Used for | What it does |
|---|---|---|
| `clean_for_lr()` | TF-IDF + Logistic Regression | Aggressive: removes stopwords, strips special chars, keeps letters only |
| `clean_for_bert()` | BERTweet / Claude LLM | Light: preserves natural language, replaces @mentions with @user token |

**Why the difference matters:**
BERTweet was pre-trained on raw tweets — over-cleaning it (e.g. removing stopwords) actually hurts performance because the model learned from natural tweet language including words like "not", "never", "but".

In [7]:
# ── CELL 6: Define cleaning functions ────────────────────────────

# Stopwords — hardcoded to avoid NLTK download issues
STOPWORDS = {
    'i','me','my','myself','we','our','you','your','he','him','his',
    'she','her','it','its','they','them','their','what','which','who',
    'this','that','am','is','are','was','were','be','been','have','has',
    'had','do','does','did','a','an','the','and','but','if','or','of',
    'at','by','for','with','to','from','in','out','on','not','no','so','rt'
}


def clean_for_lr(text: str) -> str:
    """
    Aggressive cleaning for TF-IDF + Logistic Regression.
    Removes URLs, mentions, hashtag symbols, special chars, stopwords.
    Keeps only letters, collapses whitespace.
    """
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)       # remove URLs
    text = re.sub(r'@\w+', '', text)                  # remove @mentions
    text = re.sub(r'#(\w+)', r'\1', text)             # #hashtag → hashtag (keep word)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)         # keep letters only
    text = re.sub(r'\s+', ' ', text).strip()          # collapse whitespace
    words = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return ' '.join(words)


def clean_for_bert(text: str) -> str:
    """
    Light cleaning for transformer models (BERTweet, Claude).
    Follows BERTweet preprocessing convention:
    - URLs → 'http' token
    - @mentions → '@user' token
    - Keeps punctuation, stopwords, natural language structure
    """
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\S+', 'http', text)   # URL → 'http'
    text = re.sub(r'@\w+', '@user', text)             # mention → '@user'
    text = text.encode('ascii', 'ignore').decode('ascii')  # remove non-ASCII
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Quick test
test = "Absolutely gutted for England #ThreeLions @England http://t.co/abc123 NOT BAD though!"
print(f'Original:     {test}')
print(f'For LR:       {clean_for_lr(test)}')
print(f'For BERT:     {clean_for_bert(test)}')

Original:     Absolutely gutted for England #ThreeLions @England http://t.co/abc123 NOT BAD though!
For LR:       absolutely gutted england threelions bad though
For BERT:     Absolutely gutted for England #ThreeLions @user http NOT BAD though!


In [8]:
# ── CELL 7: Apply cleaning to training data ───────────────────────
print('Cleaning training data (may take 1-2 minutes)...')

df['text_lr']   = df['text'].apply(clean_for_lr)
df['text_bert'] = df['text'].apply(clean_for_bert)

# Remove rows that are empty after cleaning
before = len(df)
df = df[df['text_lr'].str.split().str.len() >= 3]
print(f'Removed {before - len(df):,} rows that were too short after cleaning')
print(f'Remaining: {len(df):,} tweets')

print('\nSample cleaned tweets:')
for _, row in df.sample(3, random_state=42).iterrows():
    lbl = label_names.get(row['label_3class'], '?')
    print(f'  [{lbl}]')
    print(f'    Original: {row["text"][:80]}')
    print(f'    LR clean: {row["text_lr"][:80]}')
    print()

Cleaning training data (may take 1-2 minutes)...


Removed 166 rows that were too short after cleaning
Remaining: 57,597 tweets

Sample cleaned tweets:
  [Negative]
    Original: China websites blocks searches for “Fatty Kim the Third.” Many in China uae the 
    LR clean: china websites blocks searches fatty kim third many china uae term describe rotu

  [Neutral]
    Original: @user @user @user @user invitation and so did the majority. Islam is by the hear
    LR clean: invitation majority islam heart may allah guide

  [Negative]
    Original: Sam smith will be here Saturday and I'm not going....something is not right.
    LR clean: sam smith will here saturday going something right



In [9]:
# ── CELL 8: Binary train/test split ──────────────────────────────
df_binary = df.dropna(subset=['sentiment']).copy()
df_binary['sentiment'] = df_binary['sentiment'].astype(int)

X_bin = list(map(str, df_binary['text_lr']))
y_bin = list(map(int, df_binary['sentiment']))

X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_bin, y_bin,
    test_size    = 0.20,
    stratify     = y_bin,
    random_state = 42
)

y_train_bin = np.array(y_train_bin)
y_test_bin  = np.array(y_test_bin)

print('Binary split (positive vs negative):')
print(f'  Train: {len(X_train_bin):,}  '
      f'(pos={sum(y_train_bin==1):,}, neg={sum(y_train_bin==0):,})')
print(f'  Test:  {len(X_test_bin):,}   '
      f'(pos={sum(y_test_bin==1):,},  neg={sum(y_test_bin==0):,})')

Binary split (positive vs negative):
  Train: 24,928  (pos=16,105, neg=8,823)
  Test:  6,232   (pos=4,026,  neg=2,206)


In [10]:
# ── CELL 9: Three-class train/test split ─────────────────────────
X_3cls = list(map(str, df['text_lr']))
y_3cls = list(map(int, df['label_3class']))

X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(
    X_3cls, y_3cls,
    test_size    = 0.20,
    stratify     = y_3cls,
    random_state = 42
)

y_train_3 = np.array(y_train_3)
y_test_3  = np.array(y_test_3)

print('Three-class split (negative / neutral / positive):')
for split_name, X, y in [('Train', X_train_3, y_train_3), ('Test', X_test_3, y_test_3)]:
    print(f'  {split_name}: {len(X):,}')
    for lbl, name in label_names.items():
        print(f'    {name}: {sum(y==lbl):,}')

Three-class split (negative / neutral / positive):
  Train: 46,077
    Negative: 8,823
    Neutral: 21,149


    Positive: 16,105
  Test: 11,520
    Negative: 2,206
    Neutral: 5,288
    Positive: 4,026


## Step 4 — Load and Clean Euro 2024 Domain Data

In [11]:
# ── CELL 10: Load Euro 2024 tweets ───────────────────────────────
df_euro = pd.read_csv('data/euro2024_domain.csv')
df_euro['date'] = pd.to_datetime(df_euro['date'])

print(f'Euro 2024 tweets: {len(df_euro):,}')
print(f'Date range: {df_euro["date"].min().date()} → {df_euro["date"].max().date()}')
print(f'\nMatch distribution:')
print(df_euro['nearest_match'].value_counts().to_string())

# Apply same cleaning
df_euro['text_lr']   = df_euro['text'].apply(clean_for_lr)
df_euro['text_bert'] = df_euro['text'].apply(clean_for_bert)

# Remove very short tweets after cleaning
df_euro = df_euro[df_euro['text_lr'].str.split().str.len() >= 3]
print(f'\nAfter cleaning: {len(df_euro):,} usable tweets')

Euro 2024 tweets: 988
Date range: 2024-07-07 → 2024-07-21

Match distribution:
nearest_match
Tournament window              597
FIN: England 1-2 Spain         306
SF: England 2-1 Netherlands     85

After cleaning: 879 usable tweets


## Step 5 — Save Cleaned Data

In [12]:
# ── CELL 11: Save outputs ─────────────────────────────────────────
import os
os.makedirs('data', exist_ok=True)

# Save cleaned training data
df.to_csv('data/tweeteval_clean.csv', index=False)
print(f'Saved: data/tweeteval_clean.csv ({len(df):,} rows)')

# Save cleaned Euro 2024 data
df_euro.to_csv('data/euro2024_clean.csv', index=False)
print(f'Saved: data/euro2024_clean.csv ({len(df_euro):,} rows)')

print('\nColumns in tweeteval_clean.csv:')
print(list(df.columns))

print('\nColumns in euro2024_clean.csv:')
print(list(df_euro.columns))

print('\nPreprocessing complete. Run 02_baseline_models.ipynb next.')

Saved: data/tweeteval_clean.csv (57,597 rows)
Saved: data/euro2024_clean.csv (879 rows)

Columns in tweeteval_clean.csv:
['text', 'label_3class', 'sentiment', 'source', 'text_lr', 'text_bert']

Columns in euro2024_clean.csv:
['date', 'time', 'text', 'nearest_match', 'match_result', 'source', 'text_lr', 'text_bert']

Preprocessing complete. Run 02_baseline_models.ipynb next.


---
## Summary

| Step | Input | Output |
|---|---|---|
| Load | TweetEval (raw) | 57,763 labelled tweets |
| Clean (LR) | Raw text | Stopwords removed, letters only |
| Clean (BERT) | Raw text | URL/mention tokens, natural language |
| Binary split | 31,289 tweets (pos/neg) | 80/20 train/test |
| 3-class split | 57,598 tweets (all) | 80/20 train/test |
| Euro 2024 | 871 domain tweets | Cleaned, ready for model scoring |

**Next:** Open `02_baseline_models.ipynb` to train and evaluate the LR baseline and VADER.